# Step 42 — Exact source-index comparison

Read the separate step-40 and selected step-41 CSVs; do not merge origins or alter inputs. Exact equality is tested on origin-time **strings**, latitude, longitude, depth_km and magnitude after dropping duplicate five-field rows within each source. This is intentionally stricter than a physical-event match: timestamp formatting, rounding, solution revisions, different coverage and clock conventions can all produce nonmatches.

Pandas merge semantics can match missing values in join keys; an exact match does not guarantee complete metadata. Source provenance and quality fields are not compared. The two output difference CSVs contain the five comparison fields, not full provenance; look back to the source indexes/raw lines to investigate.

`out_prefix` selects the output parent directory; filenames remain fixed as `42_pinaall_not_in_40.csv` and `42_40_not_in_pinaall.csv`. Outputs overwrite existing files. For annotated PINAALL, explicitly select its separate CSV and resolve time policies first. Step 43 provides a tolerant comparison, not a reason to force exact agreement.

## 1. Configuration

Review source paths, selected PINAALL variant and output paths before running. Configuration does not execute the workflow.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    hypo40=P['SUMMARY_FILE_40_INDEX'],
    hypo41=P['SUMMARY_FILE_41_INDEX'],
    out_prefix=P['COMPARE_PREFIX'],
)
display(SETTINGS)

Configuration: /Users/thompsong/Developer/Pinatubo1991SeismicData/10_conversion/conversion_config.json
External outputs: /Users/thompsong/work/Pinatubo1991Processing/converted


{'hypo40': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/40_summary_file_index.csv',
 'hypo41': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/41_summary_file_index.csv',
 'out_prefix': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/42_pinaalldat_vs_pinatuboallsum'}

## 2. Implementation

The complete step-specific workflow is defined below. Definitions alone do not read sources or write products.

In [2]:
from pathlib import Path
import pandas as pd

COMPARE_COLS = [
    "origin_time",
    "latitude",
    "longitude",
    "depth_km",
    "magnitude",
]

def run(
    *,
    hypo40,
    hypo41,
    out_prefix
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        hypo40=hypo40,
        hypo41=hypo41,
        out_prefix=out_prefix,
    )

    df40 = pd.read_csv(args.hypo40)[COMPARE_COLS].drop_duplicates()
    df41 = pd.read_csv(args.hypo41)[COMPARE_COLS].drop_duplicates()

    merged = df41.merge(df40, how="left", on=COMPARE_COLS, indicator=True)
    pinaall_not_in_40 = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

    merged_rev = df40.merge(df41, how="left", on=COMPARE_COLS, indicator=True)
    hypo40_not_in_pinaall = merged_rev[merged_rev["_merge"] == "left_only"].drop(columns="_merge")

    out_dir = Path(args.out_prefix).parent
    out_dir.mkdir(parents=True, exist_ok=True)

    out1 = out_dir / "42_pinaall_not_in_40.csv"
    out2 = out_dir / "42_40_not_in_pinaall.csv"

    pinaall_not_in_40.to_csv(out1, index=False)
    hypo40_not_in_pinaall.to_csv(out2, index=False)

    print("\nHYPOCENTER INDEX COMPARISON (EXACT MATCH)")
    print("========================================")
    print(f"Rows in 40: {len(df40)}")
    print(f"Rows in 41: {len(df41)}\n")
    print(f"PINAALL rows found in 40: {len(df41) - len(pinaall_not_in_40)}")
    print(f"PINAALL rows NOT in 40:   {len(pinaall_not_in_40)}\n")
    print(f"40 rows NOT in PINAALL:   {len(hypo40_not_in_pinaall)}\n")
    print(f"Wrote: {out1}")
    print(f"Wrote: {out2}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Execute when ready

This cell reads configured inputs and replaces configured outputs. Inspect summaries and diagnostic tables after the run.

In [3]:
result = run(**SETTINGS)


HYPOCENTER INDEX COMPARISON (EXACT MATCH)
Rows in 40: 4628
Rows in 41: 3598

PINAALL rows found in 40: 244
PINAALL rows NOT in 40:   3354

40 rows NOT in PINAALL:   4384

Wrote: /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/42_pinaall_not_in_40.csv
Wrote: /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/42_40_not_in_pinaall.csv


In [4]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


df40


,origin_time,latitude,longitude,depth_km,magnitude
0,1991-05-07T03:21:14.290000,15.168667,120.322333,3.28,1.36
1,1991-05-07T12:52:29.950000,15.169833,120.313500,1.89,0.70
2,1991-05-08T02:37:00.060000,15.163333,120.331167,4.55,1.00
3,1991-05-08T17:56:33.040000,15.158000,120.317833,4.53,1.04
4,1991-05-08T17:59:23.980000,15.154667,120.330167,4.07,0.49


df41


,origin_time,latitude,longitude,depth_km,magnitude
0,1991-05-10T04:10:52.990000,15.440667,120.380500,15.00,0.00
1,1991-05-10T05:15:26.520000,15.155667,120.326000,3.37,0.93
2,1991-05-11T21:18:53.740000,15.152833,120.321833,2.53,0.88
3,1991-05-12T02:04:25.200000,15.166833,120.328167,3.29,0.79
4,1991-05-16T18:52:22.920000,15.134667,120.339833,1.35,0.68


merged


,origin_time,latitude,longitude,depth_km,magnitude,_merge
0,1991-05-10T04:10:52.990000,15.440667,120.380500,15.00,0.00,left_only
1,1991-05-10T05:15:26.520000,15.155667,120.326000,3.37,0.93,left_only
2,1991-05-11T21:18:53.740000,15.152833,120.321833,2.53,0.88,left_only
3,1991-05-12T02:04:25.200000,15.166833,120.328167,3.29,0.79,left_only
4,1991-05-16T18:52:22.920000,15.134667,120.339833,1.35,0.68,left_only
